# Landing Zone — Binance Crypto OHLCV Fetcher

Fetches daily OHLCV candles for each symbol in `WATCHLIST` from the
[Binance public Klines API](https://binance-docs.github.io/apidocs/spot/en/#kline-candlestick-data)
and saves **one CSV per symbol** to `/home/landing/crypto/daily/`.

**No API key required** — the Binance public API is freely accessible.

**Parameters (Cell 2):**
- `WATCHLIST` — Binance trading pairs (e.g. `BTCUSDT`)
- `DAYS` — number of past daily candles to fetch per symbol (max 1000 ≈ 2.7 years)

**Cost:** 1 API call per symbol regardless of `DAYS`.

**No Spark required** — pure Python, independent of the Spark cluster.

In [24]:
WATCHLIST = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "XRPUSDT", "ADAUSDT"]
DAYS      = 365   # daily candles per symbol — max 1000 (~2.7 years)

## Setup

In [25]:
import os

OUTPUT_DIR = "/home/landing/crypto/daily"
BASE_URL   = "https://api.binance.com/api/v3/klines"

if not 1 <= DAYS <= 1000:
    raise ValueError(f"DAYS must be between 1 and 1000, got: {DAYS}")

print(f"Watchlist  : {WATCHLIST}")
print(f"Days       : {DAYS} daily candles per symbol")
print(f"Total calls: {len(WATCHLIST)}  (1 per symbol)")
print(f"Output dir : {OUTPUT_DIR}")

Watchlist  : ['BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'XRPUSDT', 'ADAUSDT']
Days       : 365 daily candles per symbol
Total calls: 5  (1 per symbol)
Output dir : /home/landing/crypto/daily


## Fetch & Save Functions

In [26]:
import csv
import requests
from datetime import datetime, timezone
from pathlib import Path


def fetch_candles(symbol: str, days: int) -> list[dict] | None:
    """
    Fetches up to `days` daily OHLCV candles for the given symbol from Binance.
    Returns a list of flat dicts sorted oldest-first, or None on error.
    """
    params = {
        "symbol":   symbol,
        "interval": "1d",
        "limit":    days,
    }

    resp = requests.get(BASE_URL, params=params, timeout=15)

    if resp.status_code == 400:
        detail = resp.json().get("msg", resp.text)
        print(f"  -> 400 Bad Request: {detail}")
        return None
    if resp.status_code == 404:
        print(f"  -> 404 — symbol {symbol!r} not recognised.")
        return None

    resp.raise_for_status()
    raw = resp.json()

    if not raw:
        print("  -> No candles returned.")
        return None

    # Binance kline format: [openTimeMs, open, high, low, close, volume, ...]
    candles = []
    for k in raw:
        candles.append({
            "symbol": symbol,
            "date":   datetime.fromtimestamp(k[0] / 1000, tz=timezone.utc).strftime("%Y-%m-%d"),
            "open":   k[1],
            "high":   k[2],
            "low":    k[3],
            "close":  k[4],
            "volume": k[5],
        })

    return candles  # Binance returns oldest-first already


def save_csv(candles: list[dict], symbol: str, output_dir: str) -> str:
    """Saves all candles for a symbol to a single CSV. Returns path written."""
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    filepath = os.path.join(output_dir, f"{symbol}.csv")
    fieldnames = ["symbol", "date", "open", "high", "low", "close", "volume"]
    with open(filepath, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(candles)
    return filepath


print("Functions defined.")

Functions defined.


## Fetch All Symbols

In [27]:
import time

RATE_LIMIT_SLEEP = 0.5  # seconds between symbols — Binance public API is generous

results = []
skipped = []
errors  = []

print(f"Fetching {len(WATCHLIST)} symbol(s), {DAYS} candles each...\n")

for i, symbol in enumerate(WATCHLIST):
    print(f"[{i+1}/{len(WATCHLIST)}] {symbol} ", end="", flush=True)
    try:
        candles = fetch_candles(symbol, DAYS)
        if candles:
            path = save_csv(candles, symbol, OUTPUT_DIR)
            results.append({
                "symbol":  symbol,
                "candles": len(candles),
                "from":    candles[0]["date"],
                "to":      candles[-1]["date"],
                "file":    path,
            })
            print(f"-> {len(candles)} candles  ({candles[0]['date']} -> {candles[-1]['date']})  saved.")
        else:
            skipped.append(symbol)
    except Exception as exc:
        print(f"-> ERROR: {exc}")
        errors.append(symbol)

    if i < len(WATCHLIST) - 1:
        time.sleep(RATE_LIMIT_SLEEP)

print(f"\nDone. {len(results)} saved, {len(skipped)} skipped, {len(errors)} errors.")

Fetching 5 symbol(s), 365 candles each...

[1/5] BTCUSDT -> 365 candles  (2025-03-02 -> 2026-03-01)  saved.
[2/5] ETHUSDT -> 365 candles  (2025-03-02 -> 2026-03-01)  saved.
[3/5] SOLUSDT -> 365 candles  (2025-03-02 -> 2026-03-01)  saved.
[4/5] XRPUSDT -> 365 candles  (2025-03-02 -> 2026-03-01)  saved.
[5/5] ADAUSDT -> 365 candles  (2025-03-02 -> 2026-03-01)  saved.

Done. 5 saved, 0 skipped, 0 errors.


## Summary

In [28]:
import pandas as pd

if results:
    df = pd.DataFrame(results)
    print(f"Fetched {len(results)}/{len(WATCHLIST)} symbols")
    print(f"Files written to: {OUTPUT_DIR}\n")
    display(df)
else:
    print("No data saved. Check the messages above for details.")

Fetched 5/5 symbols
Files written to: /home/landing/crypto/daily



,symbol,candles,from,to,file
0,BTCUSDT,365,2025-03-02,2026-03-01,/home/landing/crypto/daily/BTCUSDT.csv
1,ETHUSDT,365,2025-03-02,2026-03-01,/home/landing/crypto/daily/ETHUSDT.csv
2,SOLUSDT,365,2025-03-02,2026-03-01,/home/landing/crypto/daily/SOLUSDT.csv
3,XRPUSDT,365,2025-03-02,2026-03-01,/home/landing/crypto/daily/XRPUSDT.csv
4,ADAUSDT,365,2025-03-02,2026-03-01,/home/landing/crypto/daily/ADAUSDT.csv
